# Applied AI Midterm — Kaggle: Classifier B and final evaluation

This notebook covers Classifier B and the final tests. Requires (1) public `blourdhuraju/dogs-vs-cats-classification`, (2) a **completed epoch-150 checkpoint dataset** containing `project/data/dataset_split.csv`, `project/checkpoints/model_A/model_A_final.keras`, and `project/checkpoints/srgan/groupnorm_v2/best_generators/generator_best_psnr.keras`. Change the owner-specific Kaggle dataset handle in these cells for independent reproduction.

**VERY IMPORTANT:** These long training cells will update the private checkpoint dataset. Run in sequence, and refresh the mounted snapshot after upload before new sessions. Do not upload checkpoints/models to a public GitHub repository without checking their permissions.

## 1. Deterministic data pipeline and frozen SRGAN generator
Classifier B sees the same image IDs/labels as Classifier A, but inputs are downsampled to 32×32 and reconstructed to 128×128 using generator epoch 107.

In [ ]:

# ============================================================
# CLASSIFIER B - RESTORE ENVIRONMENT AFTER KAGGLE RESTART
# No model training in this cell
# ============================================================

import io
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from PIL import Image, ImageOps

SEED = 42
BATCH_SIZE = 16

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

DATASET_ROOT = Path(
    "/kaggle/input/datasets/blourdhuraju/"
    "dogs-vs-cats-classification/"
    "dogs-vs-cats-classification"
)

PROJECT_DIR = Path("/kaggle/working/project")

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Restore original dataset split
# ------------------------------------------------------------

df = pd.read_csv(
    BACKUP_ROOT / "data/dataset_split.csv"
)

prefix = "dogs-vs-cats-classification/"

def resolve_path(relative_path):
    path = str(relative_path).replace("\\", "/")

    if path.startswith(prefix):
        path = path[len(prefix):]

    return str(DATASET_ROOT / path)

df["image_path"] = df["relative_path"].apply(
    resolve_path
)

assert len(df) == 24930
assert df["image_path"].map(os.path.isfile).all()

train_b_df = df[df["split"] == "train"].copy()
val_b_df = df[df["split"] == "validation"].copy()
test_b_df = df[df["split"] == "test"].copy()

assert len(train_b_df) == 15705
assert len(val_b_df) == 1746
assert len(test_b_df) == 7479

print("=== CLASSIFIER B DATASET ===")
print("Training:", len(train_b_df))
print("Validation:", len(val_b_df))
print("Reserved test:", len(test_b_df))

# ------------------------------------------------------------
# 3. Register custom PixelShuffle layer
# ------------------------------------------------------------

@tf.keras.utils.register_keras_serializable(
    package="Midterm"
)
class PixelShuffle(tf.keras.layers.Layer):

    def __init__(self, scale=2, **kwargs):
        super().__init__(**kwargs)
        self.scale = int(scale)

    def call(self, inputs):
        return tf.nn.depth_to_space(
            inputs,
            block_size=self.scale
        )

    def get_config(self):
        config = super().get_config()
        config["scale"] = self.scale
        return config

# ------------------------------------------------------------
# 4. Restore selected SRGAN generator
# ------------------------------------------------------------

best_dir = (
    BACKUP_ROOT /
    "checkpoints/srgan/groupnorm_v2/best_generators"
)

tracking_path = best_dir / "best_generator_tracking.json"

with open(tracking_path) as f:
    tracking = json.load(f)

selected_epoch = int(
    tracking["best_psnr"]["epoch"]
)

assert selected_epoch == 107

sr_generator_b = tf.keras.models.load_model(
    str(best_dir / "generator_best_psnr.keras"),
    custom_objects={
        "PixelShuffle": PixelShuffle,
        "Midterm>PixelShuffle": PixelShuffle
    },
    compile=False
)

sr_generator_b.trainable = False

assert sr_generator_b.count_params() == 959427

print("\nSRGAN generator restored.")
print("Selected epoch:", selected_epoch)

# ------------------------------------------------------------
# 5. Restore original image preprocessing
# ------------------------------------------------------------

def prepare_original_image(path_value):

    if isinstance(path_value, np.ndarray):
        path_value = path_value.item()

    path = os.fsdecode(path_value)

    with Image.open(path) as img:

        img = ImageOps.exif_transpose(img)
        img = img.convert("RGB")

        img = img.resize(
            (128, 128),
            Image.Resampling.BICUBIC
        )

        buffer = io.BytesIO()
        img.save(
            buffer,
            format="JPEG",
            quality=95
        )

        buffer.seek(0)

        with Image.open(buffer) as prepared:
            result = np.asarray(
                prepared.convert("RGB"),
                dtype=np.uint8
            ).copy()

    return result


def load_lr_image(path, label):

    hr = tf.numpy_function(
        prepare_original_image,
        [path],
        tf.uint8
    )

    hr.set_shape((128, 128, 3))

    hr = tf.image.convert_image_dtype(
        hr,
        tf.float32
    )

    lr = tf.image.resize(
        hr,
        (32, 32),
        method="bicubic",
        antialias=True
    )

    lr = tf.clip_by_value(
        lr, 0.0, 1.0
    )

    return lr, tf.cast(label, tf.float32)

# ------------------------------------------------------------
# 6. Rebuild datasets
# ------------------------------------------------------------

def make_lr_dataset(dataframe, training=False):

    ds = tf.data.Dataset.from_tensor_slices((
        dataframe["image_path"].astype(str).tolist(),
        dataframe["label"].astype("float32").tolist()
    ))

    if training:
        ds = ds.shuffle(
            len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        load_lr_image,
        num_parallel_calls=4
    )

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(1)

    return ds


train_lr_b_ds = make_lr_dataset(
    train_b_df, training=True
)

val_lr_b_ds = make_lr_dataset(
    val_b_df, training=False
)

# ------------------------------------------------------------
# 7. Rebuild SRGAN image datasets
# ------------------------------------------------------------

def apply_super_resolution(lr, labels):

    sr = sr_generator_b(lr, training=False)

    sr = tf.clip_by_value(
        sr, 0.0, 1.0
    )

    return sr, labels


train_sr_b_ds = train_lr_b_ds.map(
    apply_super_resolution,
    num_parallel_calls=1
).prefetch(1)

val_sr_b_ds = val_lr_b_ds.map(
    apply_super_resolution,
    num_parallel_calls=1
).prefetch(1)

# ------------------------------------------------------------
# 8. Verify restored pipeline
# ------------------------------------------------------------

print("\n=== PIPELINE TEST ===")

for images, labels in train_sr_b_ds.take(1):

    print("Images:", images.shape)
    print("Labels:", labels.shape)

    assert tuple(images.shape) == (
        16, 128, 128, 3
    )

    assert tuple(labels.shape) == (16,)

print("\nTensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

print("\nSUCCESS: CLASSIFIER B ENVIRONMENT RESTORED!")


## 2. Train Classifier B: frozen MobileNetV2 backbone
Head-only training, early stopping and checkpoint saving; original run ended after nine epochs.

In [ ]:

# ============================================================
# CLASSIFIER B - TRANSFER LEARNING (STAGE 1)
# MobileNetV2 on SRGAN reconstructed images
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers
from pathlib import Path

SEED = 42
tf.keras.utils.set_random_seed(SEED)

PROJECT_DIR = Path("/kaggle/working/project")
B_DIR = PROJECT_DIR / "checkpoints" / "model_B"
B_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Verify input pipeline
# ------------------------------------------------------------

assert len(train_b_df) == 15705
assert len(val_b_df) == 1746
assert len(test_b_df) == 7479

assert "train_sr_b_ds" in globals()
assert "val_sr_b_ds" in globals()

sr_generator_b.trainable = False

# ------------------------------------------------------------
# 2. Training augmentation
# ------------------------------------------------------------

def augment_one_image(image):

    image = tf.image.random_flip_left_right(image)

    image = tf.image.random_brightness(
        image, max_delta=0.08
    )

    image = tf.image.random_contrast(
        image, lower=0.9, upper=1.1
    )

    return tf.clip_by_value(image, 0.0, 1.0)


def augment_sr_batch(images, labels):

    images = tf.map_fn(
        augment_one_image,
        images,
        fn_output_signature=tf.float32
    )

    return images, labels


train_b_aug_ds = train_sr_b_ds.map(
    augment_sr_batch,
    num_parallel_calls=1
).prefetch(1)

# Validation images are never augmented
val_b_eval_ds = val_sr_b_ds

# ------------------------------------------------------------
# 3. Build Classifier B
# ------------------------------------------------------------

backbone_b = tf.keras.applications.MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights="imagenet"
)

backbone_b.trainable = False

inputs_b = tf.keras.Input(
    shape=(128, 128, 3),
    name="srgan_image"
)

# Convert pixel values from [0,1] to [-1,1]
x = layers.Rescaling(
    scale=2.0,
    offset=-1.0
)(inputs_b)

x = backbone_b(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs_b = layers.Dense(
    1,
    activation="sigmoid"
)(x)

model_b = tf.keras.Model(
    inputs_b,
    outputs_b,
    name="Classifier_B_SRGAN_MobileNetV2"
)

# ------------------------------------------------------------
# 4. Verify model architecture
# ------------------------------------------------------------

total_params = model_b.count_params()

trainable_params = sum(
    int(tf.keras.backend.count_params(w))
    for w in model_b.trainable_weights
)

print("=== CLASSIFIER B ARCHITECTURE ===")
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

assert total_params == 2259265
assert trainable_params == 1281

# ------------------------------------------------------------
# 5. Compile
# ------------------------------------------------------------

model_b.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.AUC(
            name="auc"
        )
    ]
)

# ------------------------------------------------------------
# 6. Callbacks and checkpoints
# ------------------------------------------------------------

callbacks_b = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(
            B_DIR / "model_B_head_best.keras"
        ),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(
            B_DIR / "model_B_head_epoch_{epoch:02d}.keras"
        ),
        save_best_only=False,
        verbose=0
    ),

    tf.keras.callbacks.CSVLogger(
        str(B_DIR / "model_B_head_training.csv")
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.TerminateOnNaN()
]

# ------------------------------------------------------------
# 7. Train classification head
# ------------------------------------------------------------

print("\n=== TRAINING CLASSIFIER B - STAGE 1 ===")
print("Train images:", len(train_b_df))
print("Validation images:", len(val_b_df))
print("Test images reserved:", len(test_b_df))
print("SRGAN generator: epoch 107")
print("MobileNetV2 backbone: frozen")
print("Maximum epochs: 10")

history_b_head = model_b.fit(
    train_b_aug_ds,
    validation_data=val_b_eval_ds,
    epochs=10,
    callbacks=callbacks_b,
    verbose=1
)

# ------------------------------------------------------------
# 8. Save trained model
# ------------------------------------------------------------

final_head_path = (
    B_DIR / "model_B_head_final.keras"
)

model_b.save(str(final_head_path))

print("\n=== CLASSIFIER B STAGE 1 COMPLETE ===")
print("Epochs completed:", len(history_b_head.history["loss"]))
print("Best validation accuracy:",
      max(history_b_head.history["val_accuracy"]))
print("Best validation AUC:",
      max(history_b_head.history["val_auc"]))
print("Final model:", final_head_path)
print("Checkpoints:", B_DIR)


## 3. Fine-tune Classifier B and save complete experiment
The trained head is loaded, the last 30 MobileNetV2 layers are considered (BatchNorm frozen), and the SRGAN outputs are cached **before** random classifier augmentation. No test split used for tuning.

In [ ]:

# ============================================================
# CLASSIFIER B - OVERNIGHT FINE-TUNING
#
# 1. Save Stage 1 to private Kaggle dataset
# 2. Cache SRGAN images during the first epoch
# 3. Fine-tune last 30 MobileNetV2 layers
# 4. Save best/latest checkpoints after every epoch
# 5. Upload cloud backup after every epoch
# 6. Stop safely after approximately 5.5 hours
# ============================================================

import os
import json
import time
import shutil
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import tensorflow as tf
import kagglehub

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SEED = 42
MAX_FINETUNE_EPOCHS = 20
MAX_RUN_HOURS = 5.5

job_start = time.monotonic()

SOURCE_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

PROJECT_DIR = Path("/kaggle/working/project")

B_DIR = PROJECT_DIR / "checkpoints" / "model_B"
B_DIR.mkdir(parents=True, exist_ok=True)

UPLOAD_ROOT = Path(
    "/kaggle/working/classifier_b_cloud_upload"
)

UPLOAD_DATASET = (
    "niklasohlenroth/applied-ai-midterm-srgan-checkpoints"
)

HEAD_FINAL = B_DIR / "model_B_head_final.keras"
HEAD_BEST = B_DIR / "model_B_head_best.keras"
HEAD_CSV = B_DIR / "model_B_head_training.csv"

FT_BEST = B_DIR / "model_B_finetune_best.keras"
FT_LATEST = B_DIR / "model_B_finetune_latest.keras"
FT_FINAL = B_DIR / "model_B_finetuned_final.keras"
FT_CSV = B_DIR / "model_B_finetuning.csv"

# ------------------------------------------------------------
# 2. Verify existing Stage 1 and data
# ------------------------------------------------------------

assert tf.config.list_physical_devices("GPU"), (
    "STOP: GPU not available."
)

assert "train_sr_b_ds" in globals(), (
    "STOP: Restore Classifier B data pipeline first."
)

assert "val_sr_b_ds" in globals(), (
    "STOP: Restore Classifier B data pipeline first."
)

assert len(train_b_df) == 15705
assert len(val_b_df) == 1746
assert len(test_b_df) == 7479

for path in [HEAD_FINAL, HEAD_BEST, HEAD_CSV]:
    assert path.is_file(), f"Missing Stage 1 file: {path}"

assert not FT_CSV.exists(), (
    "Fine-tuning history already exists. "
    "Use a recovery procedure instead of restarting."
)

with open(SOURCE_ROOT / "recovery_manifest.json") as f:
    srgan_manifest = json.load(f)

assert int(srgan_manifest["completed_epoch"]) == 150

head_history = pd.read_csv(HEAD_CSV)
assert len(head_history) == 9

print("=== PRE-FLIGHT CHECK ===")
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("SRGAN: 150 epochs completed")
print("Classifier B Stage 1: 9 epochs completed")
print("Stage 1 model: verified")
print("Training images:", len(train_b_df))
print("Validation images:", len(val_b_df))

# ------------------------------------------------------------
# 3. Persistent cloud backup
# ------------------------------------------------------------

def upload_classifier_b_backup(stage, epoch_number):

    print(
        f"\nPreparing cloud backup: {stage}, "
        f"epoch {epoch_number}",
        flush=True
    )

    if UPLOAD_ROOT.exists():
        shutil.rmtree(UPLOAD_ROOT)

    target = UPLOAD_ROOT / "project"

    # Preserve the complete original SRGAN backup.
    shutil.copytree(SOURCE_ROOT, target)

    target_b = target / "checkpoints" / "model_B"
    target_b.mkdir(parents=True, exist_ok=True)

    files_to_copy = [
        HEAD_FINAL,
        HEAD_BEST,
        HEAD_CSV,
        FT_BEST,
        FT_LATEST,
        FT_FINAL,
        FT_CSV
    ]

    for source in files_to_copy:
        if source.is_file():
            shutil.copy2(
                source,
                target_b / source.name
            )

    metadata = {
        "srgan_completed_epochs": 150,
        "srgan_selected_epoch": 107,
        "classifier_b_stage": stage,
        "finetune_completed_epochs": epoch_number,
        "timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }

    (target / "classifier_b_recovery.json").write_text(
        json.dumps(metadata, indent=2)
    )

    kagglehub.dataset_upload(
        UPLOAD_DATASET,
        str(UPLOAD_ROOT),
        version_notes=(
            f"Classifier B {stage} - epoch {epoch_number}"
        )
    )

    print(
        f"CLOUD BACKUP UPLOADED: "
        f"{stage}, epoch {epoch_number}",
        flush=True
    )


# IMPORTANT: Save Stage 1 BEFORE any further training.
print("\nSaving completed Stage 1 externally...")

upload_classifier_b_backup(
    stage="stage1_complete",
    epoch_number=0
)

print("\nSTAGE 1 EXTERNAL BACKUP ACCEPTED.")

# ------------------------------------------------------------
# 4. Cache super-resolved images
# ------------------------------------------------------------

print("\nPreparing cached SRGAN datasets...")
print(
    "The first epoch will build the image cache. "
    "Later epochs should run faster."
)

# SRGAN outputs are cached in system RAM.
# Approx. 3.5 GB for training + validation images.
# The cache fills automatically during the first epoch.
train_sr_cache = train_sr_b_ds.cache()
val_sr_cache = val_sr_b_ds.cache()

def augment_image(image):

    image = tf.image.random_flip_left_right(image)

    image = tf.image.random_brightness(
        image,
        max_delta=0.08
    )

    image = tf.image.random_contrast(
        image,
        lower=0.9,
        upper=1.1
    )

    return tf.clip_by_value(image, 0.0, 1.0)


def augment_batch(images, labels):

    images = tf.map_fn(
        augment_image,
        images,
        fn_output_signature=tf.float32
    )

    return images, labels


# Shuffle cached batches on each epoch.
# Augmentation is applied AFTER caching, so it stays random.
train_ft_ds = (
    train_sr_cache
    .shuffle(
        buffer_size=128,
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .map(
        augment_batch,
        num_parallel_calls=2
    )
    .prefetch(2)
)

val_ft_ds = val_sr_cache.prefetch(2)

print("Cached pipelines configured.")

# ------------------------------------------------------------
# 5. Load Stage 1 classifier
# ------------------------------------------------------------

model_b = tf.keras.models.load_model(
    str(HEAD_FINAL),
    compile=False
)

assert model_b.count_params() == 2259265

backbone_candidates = [
    layer
    for layer in model_b.layers
    if isinstance(layer, tf.keras.Model)
    and "mobilenetv2" in layer.name.lower()
]

assert len(backbone_candidates) == 1, (
    "Could not uniquely identify MobileNetV2 backbone."
)

backbone_b = backbone_candidates[0]

# ------------------------------------------------------------
# 6. Fine-tune last 30 layers
# ------------------------------------------------------------

backbone_b.trainable = True

for layer in backbone_b.layers:
    layer.trainable = False

for layer in backbone_b.layers[-30:]:

    # BatchNormalization remains frozen.
    if not isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = True

trainable_layers = [
    layer.name for layer in backbone_b.layers
    if layer.trainable
]

print("\n=== FINE-TUNING CONFIGURATION ===")
print("Backbone:", backbone_b.name)
print("Last layers considered:", 30)
print("Actually trainable backbone layers:",
      len(trainable_layers))
print("BatchNormalization: frozen")
print("Learning rate: 0.00001")

# Same low learning rate as Classifier A fine-tuning.
model_b.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.AUC(
            name="auc"
        )
    ]
)

# ------------------------------------------------------------
# 7. Automatic cloud backup callback
# ------------------------------------------------------------

class CloudBackupCallback(tf.keras.callbacks.Callback):

    def on_epoch_end(self, epoch, logs=None):

        completed_epoch = epoch + 1

        # This callback runs after model checkpoint
        # and CSVLogger, so files are available.
        upload_classifier_b_backup(
            stage="finetuning",
            epoch_number=completed_epoch
        )

        elapsed_hours = (
            time.monotonic() - job_start
        ) / 3600.0

        print(
            f"\nTotal job runtime: "
            f"{elapsed_hours:.2f} hours",
            flush=True
        )

        if elapsed_hours >= MAX_RUN_HOURS:
            print(
                "\nTIME LIMIT REACHED. "
                "Latest checkpoint has been uploaded. "
                "Stopping safely.",
                flush=True
            )
            self.model.stop_training = True


# ------------------------------------------------------------
# 8. Training callbacks
# ------------------------------------------------------------

callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        str(FT_BEST),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        str(FT_LATEST),
        save_best_only=False,
        verbose=0
    ),

    tf.keras.callbacks.CSVLogger(
        str(FT_CSV)
    ),

    CloudBackupCallback(),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.TerminateOnNaN()
]

# ------------------------------------------------------------
# 9. Start overnight fine-tuning
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("STARTING CLASSIFIER B OVERNIGHT FINE-TUNING")
print("=" * 55)
print("Maximum epochs:", MAX_FINETUNE_EPOCHS)
print("Early stopping patience:", 5)
print("Maximum planned runtime:", MAX_RUN_HOURS, "hours")
print("Cloud backup: EVERY completed epoch")
print("Generator: SRGAN epoch 107 (frozen)")
print("Test set: untouched")
print("=" * 55)

history_b_finetune = model_b.fit(
    train_ft_ds,
    validation_data=val_ft_ds,
    epochs=MAX_FINETUNE_EPOCHS,
    callbacks=callbacks,
    verbose=2
)

# ------------------------------------------------------------
# 10. Save final selected fine-tuned model
# ------------------------------------------------------------

model_b.save(str(FT_FINAL))

completed_ft_epochs = len(
    history_b_finetune.history["loss"]
)

print("\n=== FINE-TUNING SUMMARY ===")
print("Completed epochs:", completed_ft_epochs)

if completed_ft_epochs > 0:
    print(
        "Best validation accuracy:",
        max(
            history_b_finetune.history["val_accuracy"]
        )
    )
    print(
        "Best validation AUC:",
        max(
            history_b_finetune.history["val_auc"]
        )
    )

print("Final model:", FT_FINAL)

# Final cloud backup
upload_classifier_b_backup(
    stage="finetuning_final",
    epoch_number=completed_ft_epochs
)

print("\n" + "=" * 55)
print("CLASSIFIER B FINE-TUNING FINISHED")
print("Final cloud backup submitted.")
print("=" * 55)


## 4. Independent held-out evaluation
Only after all model selections are frozen. Uses all 7,479 held-out samples, calculates A/B metrics and saves paired predicted probabilities.

In [ ]:

# ============================================================
# STEP 26 - FINAL TEST EVALUATION: CLASSIFIER A VS B
# Same 7,479 held-out images; no training or augmentation
# Intermediate predictions are saved for safe recovery
# ============================================================

import os
import io
import json
import time
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from PIL import Image, ImageOps

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

SEED = 42
BATCH_SIZE = 16
SAVE_EVERY_BATCHES = 25

PROJECT_DIR = Path("/kaggle/working/project")

BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

DATASET_ROOT = Path(
    "/kaggle/input/datasets/blourdhuraju/"
    "dogs-vs-cats-classification/"
    "dogs-vs-cats-classification"
)

RESULTS_DIR = (
    PROJECT_DIR / "results" / "classifier_comparison"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_CSV = (
    RESULTS_DIR / "test_predictions_A_vs_B.csv"
)

METRICS_CSV = (
    RESULTS_DIR / "test_metrics_A_vs_B.csv"
)

METADATA_JSON = (
    RESULTS_DIR / "test_evaluation_metadata.json"
)


def find_model(relative_path):
    candidates = [
        PROJECT_DIR / relative_path,
        BACKUP_ROOT / relative_path
    ]

    for candidate in candidates:
        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        f"Model not found: {relative_path}\n"
        "Attach the latest private Kaggle checkpoint "
        "dataset or restore your saved session."
    )


MODEL_A_PATH = find_model(
    "checkpoints/model_A/model_A_final.keras"
)

MODEL_B_PATH = find_model(
    "checkpoints/model_B/model_B_finetuned_final.keras"
)

GENERATOR_PATH = find_model(
    "checkpoints/srgan/groupnorm_v2/"
    "best_generators/generator_best_psnr.keras"
)

SPLIT_PATH = find_model(
    "data/dataset_split.csv"
)

print("=== MODEL PATHS ===")
print("Classifier A:", MODEL_A_PATH)
print("Classifier B:", MODEL_B_PATH)
print("Generator:", GENERATOR_PATH)

# ------------------------------------------------------------
# 2. Load reserved test split
# ------------------------------------------------------------

df = pd.read_csv(SPLIT_PATH)

assert len(df) == 24930

test_df = (
    df[df["split"] == "test"]
    .sort_values("relative_path")
    .reset_index(drop=True)
    .copy()
)

assert len(test_df) == 7479
assert not test_df["relative_path"].duplicated().any()
assert set(test_df["label"].unique()) == {0, 1}

def resolve_path(relative_path):
    relative_path = str(relative_path).replace("\\", "/")
    prefix = "dogs-vs-cats-classification/"

    if relative_path.startswith(prefix):
        relative_path = relative_path[len(prefix):]

    return str(DATASET_ROOT / relative_path)

test_df["image_path"] = (
    test_df["relative_path"].apply(resolve_path)
)

assert test_df["image_path"].map(os.path.isfile).all(), (
    "Some source image files are missing."
)

test_paths = test_df["image_path"].tolist()
test_labels = test_df["label"].astype(int).to_numpy()
test_names = test_df["relative_path"].tolist()

print("\nTest images:", len(test_df))
print("Cats:", int((test_labels == 0).sum()))
print("Dogs:", int((test_labels == 1).sum()))

# ------------------------------------------------------------
# 3. Register SRGAN custom layer
# ------------------------------------------------------------

@tf.keras.utils.register_keras_serializable(
    package="Midterm"
)
class PixelShuffle(tf.keras.layers.Layer):

    def __init__(self, scale=2, **kwargs):
        super().__init__(**kwargs)
        self.scale = int(scale)

    def call(self, inputs):
        return tf.nn.depth_to_space(
            inputs,
            block_size=self.scale
        )

    def get_config(self):
        config = super().get_config()
        config["scale"] = self.scale
        return config

# ------------------------------------------------------------
# 4. Load completed models
# ------------------------------------------------------------

print("\nLoading completed models...")

model_a_test = tf.keras.models.load_model(
    str(MODEL_A_PATH),
    compile=False
)

model_b_test = tf.keras.models.load_model(
    str(MODEL_B_PATH),
    compile=False
)

generator_test = tf.keras.models.load_model(
    str(GENERATOR_PATH),
    custom_objects={
        "PixelShuffle": PixelShuffle,
        "Midterm>PixelShuffle": PixelShuffle
    },
    compile=False
)

model_a_test.trainable = False
model_b_test.trainable = False
generator_test.trainable = False

assert model_a_test.count_params() == 2259265
assert model_b_test.count_params() == 2259265
assert generator_test.count_params() == 959427

print("All models loaded successfully.")

# ------------------------------------------------------------
# 5. Same preprocessing as training
# ------------------------------------------------------------

def prepare_original_image_test(path_value):

    if isinstance(path_value, np.ndarray):
        path_value = path_value.item()

    path = os.fsdecode(path_value)

    with Image.open(path) as img:

        img = ImageOps.exif_transpose(img)
        img = img.convert("RGB")
        img = img.resize(
            (128, 128),
            Image.Resampling.BICUBIC
        )

        buffer = io.BytesIO()

        img.save(
            buffer,
            format="JPEG",
            quality=95
        )

        buffer.seek(0)

        with Image.open(buffer) as prepared:
            result = np.asarray(
                prepared.convert("RGB"),
                dtype=np.uint8
            ).copy()

    return result


def load_test_image(path, label):

    image = tf.numpy_function(
        prepare_original_image_test,
        [path],
        tf.uint8
    )

    image.set_shape((128, 128, 3))

    image = tf.image.convert_image_dtype(
        image,
        tf.float32
    )

    return image, tf.cast(label, tf.int32)


test_ds = tf.data.Dataset.from_tensor_slices((
    test_paths,
    test_labels.astype("int32")
))

test_ds = (
    test_ds
    .map(load_test_image, num_parallel_calls=4)
    .batch(BATCH_SIZE)
    .prefetch(1)
)

# ------------------------------------------------------------
# 6. Recover intermediate predictions if available
# ------------------------------------------------------------

rows = []

if PREDICTIONS_CSV.exists():

    previous = pd.read_csv(PREDICTIONS_CSV)

    expected_columns = [
        "relative_path",
        "label",
        "p_dog_A",
        "p_dog_B"
    ]

    assert list(previous.columns) == expected_columns

    done = len(previous)

    assert done <= len(test_df)
    assert (
        done % BATCH_SIZE == 0 or done == len(test_df)
    ), "Existing results end in a partial batch."

    assert (
        previous["relative_path"].tolist()
        == test_names[:done]
    ), "Existing prediction order does not match test data."

    assert np.array_equal(
        previous["label"].to_numpy(dtype=int),
        test_labels[:done]
    )

    rows = previous.to_dict("records")

    print("\nRecovered predictions:", done)

else:
    done = 0
    print("\nNo previous predictions: starting from zero.")

# ------------------------------------------------------------
# 7. Predict A and B on identical test images
# ------------------------------------------------------------

def save_predictions():
    temporary = PREDICTIONS_CSV.with_suffix(".tmp")

    pd.DataFrame(
        rows,
        columns=[
            "relative_path",
            "label",
            "p_dog_A",
            "p_dog_B"
        ]
    ).to_csv(temporary, index=False)

    os.replace(temporary, PREDICTIONS_CSV)


start_time = time.monotonic()
total_batches = int(np.ceil(len(test_df) / BATCH_SIZE))
completed_batches = done // BATCH_SIZE

print("\n" + "=" * 55)
print("FINAL HELD-OUT TEST EVALUATION")
print("=" * 55)
print("Already evaluated:", done)
print("Remaining images:", len(test_df) - done)
print("Total batches:", total_batches)
print("Intermediate save: every 25 batches")
print("No training / no test-time augmentation")
print("=" * 55)

if done < len(test_df):

    remaining_ds = test_ds.skip(completed_batches)

    for batch_index, (hr_images, labels) in enumerate(
        remaining_ds,
        start=completed_batches
    ):

        # Classifier A: original 128x128 image
        pred_a = model_a_test(
            hr_images,
            training=False
        ).numpy().reshape(-1)

        # Classifier B: 128 -> 32 -> SRGAN 128
        lr_images = tf.image.resize(
            hr_images,
            (32, 32),
            method="bicubic",
            antialias=True
        )

        lr_images = tf.clip_by_value(
            lr_images,
            0.0,
            1.0
        )

        sr_images = generator_test(
            lr_images,
            training=False
        )

        sr_images = tf.clip_by_value(
            sr_images,
            0.0,
            1.0
        )

        pred_b = model_b_test(
            sr_images,
            training=False
        ).numpy().reshape(-1)

        labels_np = labels.numpy()

        for i in range(len(labels_np)):

            index = len(rows)

            assert int(labels_np[i]) == int(
                test_labels[index]
            )

            rows.append({
                "relative_path": test_names[index],
                "label": int(labels_np[i]),
                "p_dog_A": float(pred_a[i]),
                "p_dog_B": float(pred_b[i])
            })

        current_batch = batch_index + 1

        if (
            current_batch % SAVE_EVERY_BATCHES == 0
            or len(rows) == len(test_df)
        ):

            save_predictions()

            elapsed_min = (
                time.monotonic() - start_time
            ) / 60

            print(
                f"Batch {current_batch}/{total_batches} "
                f"| Images {len(rows)}/{len(test_df)} "
                f"| {elapsed_min:.1f} min "
                f"| Saved",
                flush=True
            )

assert len(rows) == len(test_df)
save_predictions()

# ------------------------------------------------------------
# 8. Calculate final metrics
# ------------------------------------------------------------

pred_df = pd.read_csv(PREDICTIONS_CSV)

y_true = pred_df["label"].to_numpy(dtype=int)

metrics_rows = []

for name, column in [
    ("Classifier A - Original HR", "p_dog_A"),
    ("Classifier B - SRGAN", "p_dog_B")
]:

    y_prob = pred_df[column].to_numpy(dtype=float)
    y_pred = (y_prob >= 0.5).astype(int)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    metrics_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "auc": roc_auc_score(y_true, y_prob),
        "TN": int(cm[0, 0]),
        "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]),
        "TP": int(cm[1, 1])
    })

metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(METRICS_CSV, index=False)

metadata = {
    "dataset": "Dogs vs Cats Classification",
    "test_images": len(test_df),
    "training_images": 15705,
    "validation_images": 1746,
    "labels": {"0": "cat", "1": "dog"},
    "classifier_A": MODEL_A_PATH.name,
    "classifier_B": MODEL_B_PATH.name,
    "generator": GENERATOR_PATH.name,
    "generator_selected_epoch": 107,
    "input_A": "original 128x128",
    "input_B": "bicubic 32x32 to SRGAN 128x128",
    "threshold": 0.5,
    "random_test_augmentation": False
}

METADATA_JSON.write_text(
    json.dumps(metadata, indent=2)
)

print("\n" + "=" * 65)
print("FINAL TEST RESULTS")
print("=" * 65)

print(
    metrics_df[
        ["model", "accuracy", "precision",
         "recall", "f1", "auc"]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)

print("\nSaved files:")
print(PREDICTIONS_CSV)
print(METRICS_CSV)
print(METADATA_JSON)

# ------------------------------------------------------------
# 9. Upload complete backup with evaluation results
# ------------------------------------------------------------

import kagglehub

UPLOAD_ROOT = Path(
    "/kaggle/working/classifier_test_cloud_upload"
)

if UPLOAD_ROOT.exists():
    shutil.rmtree(UPLOAD_ROOT)

TARGET_ROOT = UPLOAD_ROOT / "project"

# Copy previous cloud backup to preserve SRGAN and A.
shutil.copytree(
    BACKUP_ROOT,
    TARGET_ROOT
)

# Ensure all important B checkpoints are included, even
# if the attached Kaggle dataset is an older snapshot.
essential_b = [
    "model_B_head_best.keras",
    "model_B_head_final.keras",
    "model_B_head_training.csv",
    "model_B_finetune_best.keras",
    "model_B_finetune_latest.keras",
    "model_B_finetuned_final.keras",
    "model_B_finetuning.csv"
]

target_b = TARGET_ROOT / "checkpoints/model_B"
target_b.mkdir(parents=True, exist_ok=True)

for filename in essential_b:

    relative = Path("checkpoints/model_B") / filename
    source = find_model(str(relative))

    shutil.copy2(
        source,
        target_b / filename
    )

# Include complete new evaluation results.
shutil.copytree(
    RESULTS_DIR,
    TARGET_ROOT / "results/classifier_comparison",
    dirs_exist_ok=True
)

assert (
    TARGET_ROOT /
    "checkpoints/srgan/groupnorm_v2/resume/"
    "ckpt-150.index"
).is_file()

assert (
    TARGET_ROOT /
    "checkpoints/model_B/model_B_finetuned_final.keras"
).is_file()

print("\nUploading final test results to private cloud dataset...")

try:

    kagglehub.dataset_upload(
        "niklasohlenroth/applied-ai-midterm-srgan-checkpoints",
        str(UPLOAD_ROOT),
        version_notes=(
            "Completed Classifier A vs B held-out test evaluation"
        )
    )

    print("\nFINAL TEST CLOUD BACKUP UPLOADED.")

except Exception as error:

    print("\nCLOUD UPLOAD FAILED:")
    print(repr(error))
    print("Results remain saved locally at:", RESULTS_DIR)
    print("Do not reset the session before backing them up.")

print("\n" + "=" * 65)
print("CLASSIFIER A VS B TEST EVALUATION COMPLETE")
print("=" * 65)


## 5. Statistics and classification figures
Paired bootstrap for accuracy difference; exact McNemar test; ROC, confusion matrices and paired correctness figures. Refresh the input dataset if these cells complain about a missing model.

In [ ]:

# ============================================================
# STEP 27 - FINAL FIGURES AND STATISTICAL COMPARISON
# No training; uses existing held-out test predictions
# ============================================================

import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

from pathlib import Path
from scipy.stats import binomtest
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix
)

PROJECT_DIR = Path("/kaggle/working/project")

BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

REL_RESULTS = Path("results/classifier_comparison")
RESULTS_DIR = PROJECT_DIR / REL_RESULTS

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PRED_FILE = "test_predictions_A_vs_B.csv"

pred_path = RESULTS_DIR / PRED_FILE

if not pred_path.exists():
    pred_path = BACKUP_ROOT / REL_RESULTS / PRED_FILE

assert pred_path.exists(), "Test predictions not found."

df = pd.read_csv(pred_path)

assert len(df) == 7479
assert not df["relative_path"].duplicated().any()
assert not df[
    ["label", "p_dog_A", "p_dog_B"]
].isna().any().any()

y_true = df["label"].to_numpy(dtype=int)

p_a = df["p_dog_A"].to_numpy(dtype=float)
p_b = df["p_dog_B"].to_numpy(dtype=float)

assert set(np.unique(y_true)) == {0, 1}
assert np.all((p_a >= 0) & (p_a <= 1))
assert np.all((p_b >= 0) & (p_b <= 1))

y_a = (p_a >= 0.5).astype(int)
y_b = (p_b >= 0.5).astype(int)

# ------------------------------------------------------------
# 1. Recalculate metrics from saved predictions
# ------------------------------------------------------------

metric_names = [
    "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"
]

def calculate_metrics(probabilities, predictions):

    return np.array([
        accuracy_score(y_true, predictions),
        precision_score(y_true, predictions),
        recall_score(y_true, predictions),
        f1_score(y_true, predictions),
        roc_auc_score(y_true, probabilities)
    ])

metrics_a = calculate_metrics(p_a, y_a)
metrics_b = calculate_metrics(p_b, y_b)

print("=== VERIFIED TEST RESULTS ===")

for name, a, b in zip(
    metric_names, metrics_a, metrics_b
):
    print(
        f"{name:12s} "
        f"A={a:.5f}  B={b:.5f}  "
        f"Difference={a-b:+.5f}"
    )

# ------------------------------------------------------------
# 2. Paired statistical comparison
# ------------------------------------------------------------

correct_a = (y_a == y_true)
correct_b = (y_b == y_true)

both_correct = int(
    np.sum(correct_a & correct_b)
)

a_only = int(
    np.sum(correct_a & ~correct_b)
)

b_only = int(
    np.sum(~correct_a & correct_b)
)

both_wrong = int(
    np.sum(~correct_a & ~correct_b)
)

assert (
    both_correct + a_only + b_only + both_wrong
) == 7479

# Exact McNemar test:
# compare cases correctly predicted by only one model.
discordant = a_only + b_only

if discordant > 0:
    mcnemar_p = binomtest(
        min(a_only, b_only),
        discordant,
        p=0.5,
        alternative="two-sided"
    ).pvalue
else:
    mcnemar_p = 1.0

# Paired bootstrap confidence interval for accuracy difference.
rng = np.random.default_rng(42)

paired_differences = (
    correct_a.astype(np.int8)
    - correct_b.astype(np.int8)
)

bootstrap_differences = []

N_BOOTSTRAP = 3000
N = len(df)

# Process in chunks to avoid large memory allocations.
for start in range(0, N_BOOTSTRAP, 100):

    count = min(
        100,
        N_BOOTSTRAP - start
    )

    indices = rng.integers(
        0, N,
        size=(count, N)
    )

    bootstrap_differences.extend(
        paired_differences[indices]
        .mean(axis=1)
        .tolist()
    )

ci_low, ci_high = np.quantile(
    bootstrap_differences,
    [0.025, 0.975]
)

accuracy_difference = (
    metrics_a[0] - metrics_b[0]
)

statistics = {
    "test_images": N,
    "both_correct": both_correct,
    "A_correct_B_wrong": a_only,
    "A_wrong_B_correct": b_only,
    "both_wrong": both_wrong,
    "accuracy_A": float(metrics_a[0]),
    "accuracy_B": float(metrics_b[0]),
    "accuracy_difference": float(
        accuracy_difference
    ),
    "accuracy_difference_95ci": [
        float(ci_low),
        float(ci_high)
    ],
    "bootstrap_resamples": N_BOOTSTRAP,
    "mcnemar_exact_p": float(mcnemar_p),
    "threshold": 0.5
}

stats_path = RESULTS_DIR / "paired_statistics.json"

stats_path.write_text(
    json.dumps(statistics, indent=2)
)

print("\n=== PAIRED STATISTICAL ANALYSIS ===")
print("Both correct:", both_correct)
print("Only A correct:", a_only)
print("Only B correct:", b_only)
print("Both wrong:", both_wrong)

print(
    "Accuracy difference: "
    f"{100*accuracy_difference:.3f} percentage points"
)

print(
    "95% paired bootstrap CI: "
    f"[{100*ci_low:.3f}, {100*ci_high:.3f}] "
    "percentage points"
)

print(f"Exact McNemar p-value: {mcnemar_p:.6g}")

# ------------------------------------------------------------
# 3. Figure folders and formatting
# ------------------------------------------------------------

VECTOR_DIR = RESULTS_DIR / "figures" / "vector"
RASTER_DIR = RESULTS_DIR / "figures" / "raster"

VECTOR_DIR.mkdir(parents=True, exist_ok=True)
RASTER_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "figure.dpi": 120,
    "savefig.dpi": 250,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

def save_figure(fig, name):

    fig.savefig(
        VECTOR_DIR / f"{name}.pdf",
        bbox_inches="tight"
    )

    fig.savefig(
        VECTOR_DIR / f"{name}.svg",
        bbox_inches="tight"
    )

    fig.savefig(
        RASTER_DIR / f"{name}.png",
        dpi=250,
        bbox_inches="tight"
    )

    plt.close(fig)

# ------------------------------------------------------------
# 4. Figure 1 - Model performance
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 4.3))

positions = np.arange(len(metric_names))

ax.scatter(
    metrics_a * 100,
    positions - 0.12,
    s=65,
    label="Classifier A - Original HR",
    marker="o"
)

ax.scatter(
    metrics_b * 100,
    positions + 0.12,
    s=65,
    label="Classifier B - SRGAN",
    marker="s"
)

for i in range(len(metric_names)):
    ax.plot(
        [metrics_b[i]*100, metrics_a[i]*100],
        [positions[i], positions[i]],
        linewidth=1,
        alpha=0.4
    )

ax.set_yticks(positions)
ax.set_yticklabels(metric_names)
ax.invert_yaxis()

ax.set_xlim(88, 100.5)
ax.set_xlabel("Test performance (%)")
ax.set_title(
    "Classifier A vs. B - Held-out Test Performance\n"
    "(Zoomed horizontal axis: 88-100%)"
)

ax.grid(axis="x", alpha=0.25)
ax.legend(loc="lower right", fontsize=8)

fig.tight_layout()
save_figure(fig, "test_performance_comparison")

# ------------------------------------------------------------
# 5. Figure 2 - ROC curves
# ------------------------------------------------------------

fpr_a, tpr_a, _ = roc_curve(y_true, p_a)
fpr_b, tpr_b, _ = roc_curve(y_true, p_b)

fig, ax = plt.subplots(figsize=(6, 5))

ax.plot(
    fpr_a,
    tpr_a,
    linewidth=2,
    label=f"Classifier A (AUC={metrics_a[4]:.4f})"
)

ax.plot(
    fpr_b,
    tpr_b,
    linewidth=2,
    label=f"Classifier B (AUC={metrics_b[4]:.4f})"
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1,
    alpha=0.5
)

ax.set(
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
    title="ROC Curves - Held-out Test Set",
    xlim=(0, 1),
    ylim=(0, 1.01)
)

ax.legend(loc="lower right")
ax.grid(alpha=0.2)

fig.tight_layout()
save_figure(fig, "test_roc_curves")

# ------------------------------------------------------------
# 6. Figure 3 - Confusion matrices
# ------------------------------------------------------------

cm_a = confusion_matrix(
    y_true, y_a, labels=[0, 1]
)

cm_b = confusion_matrix(
    y_true, y_b, labels=[0, 1]
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(9, 4)
)

max_count = max(
    cm_a.max(),
    cm_b.max()
)

for ax, cm, title in zip(
    axes,
    [cm_a, cm_b],
    ["Classifier A - Original HR",
     "Classifier B - SRGAN"]
):

    ax.imshow(
        cm,
        cmap="Blues",
        vmin=0,
        vmax=max_count
    )

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(["Cat", "Dog"])
    ax.set_yticklabels(["Cat", "Dog"])

    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(title)

    for row in range(2):
        for col in range(2):

            value = int(cm[row, col])

            ax.text(
                col,
                row,
                f"{value:,}",
                ha="center",
                va="center",
                fontsize=12,
                color=(
                    "white"
                    if value > max_count * 0.55
                    else "black"
                )
            )

fig.suptitle(
    "Confusion Matrices - 7,479 Test Images"
)

fig.tight_layout()
save_figure(fig, "test_confusion_matrices")

# ------------------------------------------------------------
# 7. Figure 4 - Paired correctness
# ------------------------------------------------------------

categories = [
    "Both correct",
    "Only A correct",
    "Only B correct",
    "Both wrong"
]

counts = [
    both_correct,
    a_only,
    b_only,
    both_wrong
]

fig, ax = plt.subplots(figsize=(7.2, 4))

bars = ax.barh(
    categories,
    counts
)

ax.invert_yaxis()

ax.set_xlabel("Number of test images")
ax.set_title(
    "Paired Prediction Agreement - Classifiers A and B"
)

for bar, count in zip(bars, counts):

    ax.text(
        bar.get_width() + 18,
        bar.get_y() + bar.get_height()/2,
        f"{count:,}",
        va="center",
        fontsize=9
    )

ax.set_xlim(0, max(counts)*1.13)
ax.grid(axis="x", alpha=0.2)

fig.tight_layout()
save_figure(fig, "paired_prediction_agreement")

# ------------------------------------------------------------
# 8. Save consolidated numerical results
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "metric": metric_names,
    "classifier_A": metrics_a,
    "classifier_B": metrics_b,
    "difference_A_minus_B": (
        metrics_a - metrics_b
    )
})

comparison.to_csv(
    RESULTS_DIR / "final_metric_comparison.csv",
    index=False
)

print("\n=== FIGURES CREATED ===")

for path in sorted(VECTOR_DIR.glob("*.pdf")):
    print(path.name)

print("\nAll figures also saved as SVG and PNG.")

# ------------------------------------------------------------
# 9. Cloud backup - preserve all previous checkpoints
# ------------------------------------------------------------

UPLOAD_ROOT = Path(
    "/kaggle/working/classifier_figures_cloud_upload"
)

TARGET_ROOT = UPLOAD_ROOT / "project"

required_model = (
    BACKUP_ROOT /
    "checkpoints/model_B/model_B_finetuned_final.keras"
)

assert required_model.is_file(), (
    "Attach the latest checkpoint dataset "
    "before uploading a new dataset version."
)

if UPLOAD_ROOT.exists():
    shutil.rmtree(UPLOAD_ROOT)

shutil.copytree(
    BACKUP_ROOT,
    TARGET_ROOT
)

shutil.copytree(
    RESULTS_DIR,
    TARGET_ROOT / REL_RESULTS,
    dirs_exist_ok=True
)

print("\nUploading scientific figures and statistics...")

try:

    kagglehub.dataset_upload(
        "niklasohlenroth/applied-ai-midterm-srgan-checkpoints",
        str(UPLOAD_ROOT),
        version_notes=(
            "Scientific figures, paired statistics and final "
            "classifier comparison"
        )
    )

    print("\nCLOUD BACKUP UPLOADED.")

except Exception as error:

    print("\nCloud upload failed:", repr(error))
    print("All results are still stored locally.")

print("\n" + "=" * 55)
print("STEP 27 COMPLETE")
print("=" * 55)


## 6. SRGAN reconstruction quality on 256 held-out samples
128 cats and 128 dogs chosen reproducibly with seed 42. Compare SRGAN with bicubic for PSNR, SSIM and qualitative examples. This sample is not used to tune the models.

In [ ]:

# ============================================================
# STEP 28 - SRGAN IMAGE QUALITY EVALUATION
# 256 held-out test images: 128 cats + 128 dogs
# Compare Bicubic and SRGAN against original HR images
# Save qualitative examples and PSNR/SSIM results
# ============================================================

import io
import os
import json
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import kagglehub

from pathlib import Path
from PIL import Image, ImageOps

SEED = 42

PROJECT_DIR = Path("/kaggle/working/project")

BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

DATASET_ROOT = Path(
    "/kaggle/input/datasets/blourdhuraju/"
    "dogs-vs-cats-classification/"
    "dogs-vs-cats-classification"
)

RESULTS_DIR = PROJECT_DIR / "results" / "srgan_quality"
VECTOR_DIR = RESULTS_DIR / "figures" / "vector"
RASTER_DIR = RESULTS_DIR / "figures" / "raster"

for folder in [RESULTS_DIR, VECTOR_DIR, RASTER_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load original test split
# ------------------------------------------------------------

split_path = BACKUP_ROOT / "data/dataset_split.csv"

if not split_path.exists():
    split_path = PROJECT_DIR / "data/dataset_split.csv"

df = pd.read_csv(split_path)

test_df = (
    df[df["split"] == "test"]
    .sort_values("relative_path")
    .reset_index(drop=True)
)

assert len(test_df) == 7479

def resolve_path(relative_path):

    relative_path = str(relative_path).replace("\\", "/")

    prefix = "dogs-vs-cats-classification/"

    if relative_path.startswith(prefix):
        relative_path = relative_path[len(prefix):]

    return DATASET_ROOT / relative_path

test_df["image_path"] = test_df[
    "relative_path"
].apply(resolve_path)

# Stratified, deterministic random sample.
rng = np.random.default_rng(SEED)

cat_ids = np.where(test_df["label"].to_numpy() == 0)[0]
dog_ids = np.where(test_df["label"].to_numpy() == 1)[0]

selected_indices = np.concatenate([
    rng.choice(cat_ids, size=128, replace=False),
    rng.choice(dog_ids, size=128, replace=False)
])

sample_df = (
    test_df.iloc[selected_indices]
    .reset_index(drop=True)
    .copy()
)

assert len(sample_df) == 256

sample_df[
    ["relative_path", "label"]
].to_csv(
    RESULTS_DIR / "quality_sample_manifest.csv",
    index=False
)

print("=== SRGAN QUALITY EVALUATION ===")
print("Selected test images:", len(sample_df))
print("Cats:", int((sample_df.label == 0).sum()))
print("Dogs:", int((sample_df.label == 1).sum()))

# ------------------------------------------------------------
# 2. Load generator
# ------------------------------------------------------------

@tf.keras.utils.register_keras_serializable(
    package="Midterm"
)
class PixelShuffle(tf.keras.layers.Layer):

    def __init__(self, scale=2, **kwargs):
        super().__init__(**kwargs)
        self.scale = int(scale)

    def call(self, inputs):
        return tf.nn.depth_to_space(
            inputs,
            block_size=self.scale
        )

    def get_config(self):
        config = super().get_config()
        config["scale"] = self.scale
        return config


generator_path = (
    BACKUP_ROOT /
    "checkpoints/srgan/groupnorm_v2/"
    "best_generators/generator_best_psnr.keras"
)

if not generator_path.exists():
    generator_path = (
        PROJECT_DIR /
        "checkpoints/srgan/groupnorm_v2/"
        "best_generators/generator_best_psnr.keras"
    )

generator_quality = tf.keras.models.load_model(
    str(generator_path),
    custom_objects={
        "PixelShuffle": PixelShuffle,
        "Midterm>PixelShuffle": PixelShuffle
    },
    compile=False
)

generator_quality.trainable = False

assert generator_quality.count_params() == 959427

# ------------------------------------------------------------
# 3. Match original preprocessing
# ------------------------------------------------------------

def load_hr_image(path):

    with Image.open(path) as img:

        img = ImageOps.exif_transpose(img)
        img = img.convert("RGB")

        img = img.resize(
            (128, 128),
            Image.Resampling.BICUBIC
        )

        buffer = io.BytesIO()

        img.save(
            buffer,
            format="JPEG",
            quality=95
        )

        buffer.seek(0)

        with Image.open(buffer) as prepared:

            result = np.asarray(
                prepared.convert("RGB"),
                dtype=np.uint8
            ).copy()

    return result.astype(np.float32) / 255.0

# ------------------------------------------------------------
# 4. Evaluate all 256 images
# ------------------------------------------------------------

quality_rows = []
visual_examples = {}

# Four cats and four dogs from the random sample.
visual_indices = [
    0, 128,
    1, 129,
    2, 130,
    3, 131
]

for start in range(0, len(sample_df), 16):

    end = min(start + 16, len(sample_df))

    batch_rows = sample_df.iloc[start:end]

    hr_np = np.stack([
        load_hr_image(path)
        for path in batch_rows["image_path"]
    ])

    hr = tf.convert_to_tensor(
        hr_np,
        dtype=tf.float32
    )

    # Original HR -> LR: 128x128 -> 32x32
    lr = tf.image.resize(
        hr,
        (32, 32),
        method="bicubic",
        antialias=True
    )

    lr = tf.clip_by_value(lr, 0.0, 1.0)

    # Bicubic baseline
    bicubic = tf.image.resize(
        lr,
        (128, 128),
        method="bicubic"
    )

    bicubic = tf.clip_by_value(
        bicubic, 0.0, 1.0
    )

    # Trained SRGAN generator
    srgan = generator_quality(
        lr,
        training=False
    )

    srgan = tf.clip_by_value(
        srgan, 0.0, 1.0
    )

    # Image quality metrics
    bicubic_psnr = tf.image.psnr(
        hr, bicubic, max_val=1.0
    ).numpy()

    srgan_psnr = tf.image.psnr(
        hr, srgan, max_val=1.0
    ).numpy()

    bicubic_ssim = tf.image.ssim(
        hr, bicubic, max_val=1.0
    ).numpy()

    srgan_ssim = tf.image.ssim(
        hr, srgan, max_val=1.0
    ).numpy()

    # Nearest-neighbor display reveals LR pixel grid.
    lr_display = tf.image.resize(
        lr,
        (128, 128),
        method="nearest"
    ).numpy()

    bicubic_np = bicubic.numpy()
    srgan_np = srgan.numpy()

    for j, (_, record) in enumerate(
        batch_rows.iterrows()
    ):

        global_idx = start + j

        quality_rows.append({
            "relative_path": record["relative_path"],
            "label": int(record["label"]),
            "bicubic_psnr": float(bicubic_psnr[j]),
            "srgan_psnr": float(srgan_psnr[j]),
            "bicubic_ssim": float(bicubic_ssim[j]),
            "srgan_ssim": float(srgan_ssim[j])
        })

        if global_idx in visual_indices:

            visual_examples[global_idx] = {
                "hr": hr_np[j],
                "lr": lr_display[j],
                "bicubic": bicubic_np[j],
                "srgan": srgan_np[j],
                "label": int(record["label"])
            }

    print(
        f"Processed {end}/{len(sample_df)} images",
        flush=True
    )

quality_df = pd.DataFrame(quality_rows)

quality_df.to_csv(
    RESULTS_DIR / "quality_per_image.csv",
    index=False
)

# ------------------------------------------------------------
# 5. Summary and paired confidence intervals
# ------------------------------------------------------------

summary = {}

for metric in ["psnr", "ssim"]:

    baseline = quality_df[
        f"bicubic_{metric}"
    ].to_numpy()

    enhanced = quality_df[
        f"srgan_{metric}"
    ].to_numpy()

    differences = enhanced - baseline

    # Paired bootstrap over the 256 sampled images.
    bootstrap_rng = np.random.default_rng(42)

    indices = bootstrap_rng.integers(
        0,
        len(differences),
        size=(2000, len(differences))
    )

    bootstrap_means = differences[indices].mean(
        axis=1
    )

    ci = np.quantile(
        bootstrap_means,
        [0.025, 0.975]
    )

    summary[metric] = {
        "bicubic_mean": float(baseline.mean()),
        "srgan_mean": float(enhanced.mean()),
        "mean_difference_srgan_minus_bicubic": float(
            differences.mean()
        ),
        "difference_95ci": [
            float(ci[0]),
            float(ci[1])
        ],
        "fraction_srgan_better": float(
            np.mean(differences > 0)
        )
    }

summary["sample_size"] = 256
summary["sample_seed"] = 42
summary["selected_generator_epoch"] = 107
summary["test_split_used_for_generator_selection"] = False

(RESULTS_DIR / "quality_summary.json").write_text(
    json.dumps(summary, indent=2)
)

print("\n=== QUALITY RESULTS ===")

for metric in ["psnr", "ssim"]:

    values = summary[metric]

    print(
        f"{metric.upper()}: "
        f"Bicubic={values['bicubic_mean']:.4f}, "
        f"SRGAN={values['srgan_mean']:.4f}, "
        f"Difference={values['mean_difference_srgan_minus_bicubic']:+.4f}"
    )

# ------------------------------------------------------------
# 6. Save figures
# ------------------------------------------------------------

plt.rcParams.update({
    "font.size": 10,
    "figure.dpi": 120,
    "savefig.dpi": 250,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

def save_figure(fig, name):

    fig.savefig(
        VECTOR_DIR / f"{name}.pdf",
        bbox_inches="tight"
    )

    fig.savefig(
        VECTOR_DIR / f"{name}.svg",
        bbox_inches="tight"
    )

    fig.savefig(
        RASTER_DIR / f"{name}.png",
        dpi=250,
        bbox_inches="tight"
    )

    plt.close(fig)

# Figure 1: Eight representative random examples.
fig, axes = plt.subplots(
    8, 4,
    figsize=(12, 20)
)

column_titles = [
    "Original HR\n128 x 128",
    "Low Resolution\n32 x 32 (nearest display)",
    "Bicubic Upscaling\n128 x 128",
    "SRGAN Upscaling\n128 x 128"
]

for row_idx, image_idx in enumerate(visual_indices):

    example = visual_examples[image_idx]

    label = (
        "Cat" if example["label"] == 0
        else "Dog"
    )

    images = [
        example["hr"],
        example["lr"],
        example["bicubic"],
        example["srgan"]
    ]

    for col_idx, image in enumerate(images):

        ax = axes[row_idx, col_idx]

        ax.imshow(
            np.clip(image, 0, 1)
        )

        ax.set_xticks([])
        ax.set_yticks([])

        if row_idx == 0:
            ax.set_title(
                column_titles[col_idx],
                fontsize=10
            )

        if col_idx == 0:
            ax.set_ylabel(
                f"{label} {row_idx + 1}",
                fontsize=10
            )

fig.suptitle(
    "Held-out Test Examples: Original vs Bicubic vs SRGAN",
    fontsize=13,
    y=0.998
)

fig.tight_layout()
save_figure(fig, "srgan_qualitative_examples")

# Figure 2: PSNR and SSIM means with paired comparison.
fig, axes = plt.subplots(
    1, 2,
    figsize=(9, 4)
)

for ax, metric, unit in zip(
    axes,
    ["psnr", "ssim"],
    ["dB", ""]
):

    values = [
        summary[metric]["bicubic_mean"],
        summary[metric]["srgan_mean"]
    ]

    bars = ax.bar(
        ["Bicubic", "SRGAN"],
        values
    )

    ax.set_title(metric.upper())
    ax.set_ylabel(unit if unit else "SSIM")

    ax.set_ylim(
        0,
        max(values) * 1.15
    )

    for bar, value in zip(bars, values):

        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height(),
            f"{value:.3f}",
            ha="center",
            va="bottom"
        )

    ax.grid(axis="y", alpha=0.2)

fig.suptitle(
    "Image Quality on 256 Held-out Test Images",
    fontsize=12
)

fig.tight_layout()
save_figure(fig, "bicubic_vs_srgan_quality")

print("\nScientific figures saved as PDF, SVG and PNG.")

# ------------------------------------------------------------
# 7. Upload without losing Step 27 results
# ------------------------------------------------------------

STEP27_SNAPSHOT = Path(
    "/kaggle/working/"
    "classifier_figures_cloud_upload/project"
)

if (
    STEP27_SNAPSHOT /
    "results/classifier_comparison/paired_statistics.json"
).is_file():

    source_snapshot = STEP27_SNAPSHOT

else:
    source_snapshot = BACKUP_ROOT

# Verify that the previous results are preserved.
required_previous = [
    "results/classifier_comparison/paired_statistics.json",
    "results/classifier_comparison/figures/vector/"
    "test_roc_curves.pdf",
    "checkpoints/model_B/model_B_finetuned_final.keras",
    "checkpoints/srgan/groupnorm_v2/resume/ckpt-150.index"
]

for relative in required_previous:

    if not (
        source_snapshot / relative
    ).is_file():

        raise FileNotFoundError(
            "Previous backup is incomplete: "
            + relative
            + ". Do not overwrite the cloud dataset."
        )

UPLOAD_ROOT = Path(
    "/kaggle/working/"
    "srgan_quality_cloud_upload"
)

if UPLOAD_ROOT.exists():
    shutil.rmtree(UPLOAD_ROOT)

TARGET_ROOT = UPLOAD_ROOT / "project"

shutil.copytree(
    source_snapshot,
    TARGET_ROOT
)

shutil.copytree(
    RESULTS_DIR,
    TARGET_ROOT / "results/srgan_quality",
    dirs_exist_ok=True
)

print("\nUploading quality results and figures...")

try:

    kagglehub.dataset_upload(
        "niklasohlenroth/applied-ai-midterm-srgan-checkpoints",
        str(UPLOAD_ROOT),
        version_notes=(
            "Held-out SRGAN quality analysis and qualitative figures"
        )
    )

    print("\nSRGAN QUALITY BACKUP UPLOADED.")

except Exception as error:

    print("\nCloud upload failed:", repr(error))
    print("All new results remain saved locally.")

print("\n" + "=" * 55)
print("STEP 28 COMPLETE")
print("=" * 55)


## 7. Training curves and final archive
Only uses training logs; does not consume GPU training quota. Both file-derived scientific figures and numerical summaries are written to `results/`.

In [ ]:

# ============================================================
# STEP 29 - TRAINING HISTORY FIGURES
# SRGAN and Classifier B training diagnostics
# No training required
# ============================================================

from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

PROJECT_DIR = Path("/kaggle/working/project")

BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

STEP28_ROOT = Path(
    "/kaggle/working/srgan_quality_cloud_upload/project"
)

# Use the latest complete backup.
required = [
    "results/srgan_quality/quality_summary.json",
    "results/classifier_comparison/paired_statistics.json",
    "checkpoints/model_B/model_B_finetuned_final.keras"
]

SOURCE_ROOT = None

for candidate in [STEP28_ROOT, BACKUP_ROOT]:

    if all((candidate / p).is_file() for p in required):
        SOURCE_ROOT = candidate
        break

if SOURCE_ROOT is None:
    raise FileNotFoundError(
        "Latest complete backup not found. "
        "Attach the newest private Kaggle dataset version."
    )

RESULTS_DIR = (
    PROJECT_DIR / "results/training_diagnostics"
)

VECTOR_DIR = RESULTS_DIR / "figures/vector"
RASTER_DIR = RESULTS_DIR / "figures/raster"

VECTOR_DIR.mkdir(parents=True, exist_ok=True)
RASTER_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load training histories
# ------------------------------------------------------------

model_dir = SOURCE_ROOT / "checkpoints/model_B"

head = pd.read_csv(
    model_dir / "model_B_head_training.csv"
)

finetune = pd.read_csv(
    model_dir / "model_B_finetuning.csv"
)

srgan = pd.read_csv(
    SOURCE_ROOT / "results/srgan_training_groupnorm_v2.csv"
)

assert len(srgan) == 150
assert len(head) == 9
assert len(finetune) == 11

required_metrics = [
    "loss", "val_loss",
    "accuracy", "val_accuracy",
    "auc", "val_auc"
]

for frame in [head, finetune]:
    assert all(
        name in frame.columns
        for name in required_metrics
    )

print("=== TRAINING HISTORIES ===")
print("SRGAN epochs:", len(srgan))
print("Classifier B head epochs:", len(head))
print("Classifier B fine-tuning epochs:", len(finetune))
print("SRGAN CSV columns:", list(srgan.columns))

# ------------------------------------------------------------
# 2. Formatting
# ------------------------------------------------------------

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "figure.dpi": 120,
    "savefig.dpi": 250,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

def save_figure(fig, filename):

    fig.savefig(
        VECTOR_DIR / f"{filename}.pdf",
        bbox_inches="tight"
    )

    fig.savefig(
        VECTOR_DIR / f"{filename}.svg",
        bbox_inches="tight"
    )

    fig.savefig(
        RASTER_DIR / f"{filename}.png",
        bbox_inches="tight",
        dpi=250
    )

    plt.close(fig)

# ------------------------------------------------------------
# 3. Classifier B - Training and validation curves
# ------------------------------------------------------------

head_epochs = np.arange(1, len(head) + 1)

ft_epochs = np.arange(
    len(head) + 1,
    len(head) + len(finetune) + 1
)

best_head_epoch = int(head["val_loss"].idxmin()) + 1
best_ft_epoch = int(finetune["val_loss"].idxmin()) + 1

fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 4.3)
)

metric_config = [
    ("accuracy", "Accuracy", "Accuracy"),
    ("loss", "Binary Cross-Entropy", "Loss"),
    ("auc", "ROC-AUC", "AUC")
]

for ax, (metric, title, ylabel) in zip(
    axes,
    metric_config
):

    ax.plot(
        head_epochs,
        head[metric],
        label="Training - Frozen backbone",
        linewidth=1.6
    )

    ax.plot(
        head_epochs,
        head["val_" + metric],
        label="Validation - Frozen backbone",
        linewidth=1.6,
        linestyle="--"
    )

    ax.plot(
        ft_epochs,
        finetune[metric],
        label="Training - Fine-tuning",
        linewidth=1.6
    )

    ax.plot(
        ft_epochs,
        finetune["val_" + metric],
        label="Validation - Fine-tuning",
        linewidth=1.6,
        linestyle="--"
    )

    ax.axvline(
        len(head) + 0.5,
        color="gray",
        linestyle=":",
        linewidth=1
    )

    ax.set_title(title)
    ax.set_xlabel("Total training epoch")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.2)

axes[0].legend(fontsize=7, loc="lower right")

fig.suptitle(
    "Classifier B - Frozen Backbone and Fine-tuning",
    fontsize=13
)

fig.tight_layout()

save_figure(
    fig,
    "classifier_B_training_history"
)

# ------------------------------------------------------------
# 4. SRGAN PSNR and SSIM histories
# ------------------------------------------------------------

srgan_epochs = np.arange(1, len(srgan) + 1)

def select_metric_columns(frame, keyword):

    matches = [
        c for c in frame.columns
        if keyword in c.lower()
    ]

    # Prefer validation metrics when available.
    matches.sort(
        key=lambda c: (
            "val" not in c.lower(),
            c.lower()
        )
    )

    return matches

psnr_columns = select_metric_columns(
    srgan, "psnr"
)

ssim_columns = select_metric_columns(
    srgan, "ssim"
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(11, 4.2)
)

for ax, columns, title, ylabel in zip(
    axes,
    [psnr_columns, ssim_columns],
    ["PSNR during SRGAN training",
     "SSIM during SRGAN training"],
    ["PSNR (dB)", "SSIM"]
):

    valid_count = 0

    for column in columns[:4]:

        values = pd.to_numeric(
            srgan[column],
            errors="coerce"
        )

        if values.notna().sum() == 0:
            continue

        ax.plot(
            srgan_epochs,
            values,
            label=column,
            linewidth=1.2
        )

        valid_count += 1

    if valid_count == 0:

        ax.text(
            0.5, 0.5,
            "Metric not found in training CSV",
            transform=ax.transAxes,
            ha="center",
            va="center"
        )

    else:
        ax.legend(fontsize=8)

    ax.axvline(
        107,
        linestyle="--",
        color="gray",
        linewidth=1.2,
        label="Selected epoch 107"
    )

    ax.set_xlim(1, 150)
    ax.set_xlabel("Adversarial training epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.2)

fig.suptitle(
    "SRGAN - Image Quality across 150 Epochs",
    fontsize=13
)

fig.tight_layout()

save_figure(
    fig,
    "srgan_training_quality_history"
)

# ------------------------------------------------------------
# 5. Save summary for scientific report
# ------------------------------------------------------------

summary = {
    "srgan_epochs": 150,
    "selected_srgan_epoch": 107,
    "classifier_B_head_epochs": len(head),
    "classifier_B_head_best_loss_epoch": best_head_epoch,
    "classifier_B_finetuning_epochs": len(finetune),
    "classifier_B_finetuning_best_loss_epoch": best_ft_epoch,
    "classifier_B_best_head_val_loss": float(
        head["val_loss"].min()
    ),
    "classifier_B_best_finetune_val_loss": float(
        finetune["val_loss"].min()
    ),
    "classifier_B_best_finetune_val_accuracy": float(
        finetune["val_accuracy"].max()
    ),
    "classifier_B_best_finetune_val_auc": float(
        finetune["val_auc"].max()
    ),
    "srgan_psnr_columns": psnr_columns,
    "srgan_ssim_columns": ssim_columns
}

(RESULTS_DIR / "training_summary.json").write_text(
    json.dumps(summary, indent=2)
)

print("\n=== TRAINING SUMMARY ===")
print("Classifier B head best epoch:", best_head_epoch)
print("Classifier B fine-tuning best epoch:", best_ft_epoch)
print(
    "Best fine-tuning validation accuracy:",
    summary["classifier_B_best_finetune_val_accuracy"]
)

print("\nFigures saved:")
for path in sorted(VECTOR_DIR.glob("*.pdf")):
    print(path.name)

# ------------------------------------------------------------
# 6. Cloud backup with all previous results
# ------------------------------------------------------------

UPLOAD_ROOT = Path(
    "/kaggle/working/training_diagnostics_cloud_upload"
)

if UPLOAD_ROOT.exists():
    shutil.rmtree(UPLOAD_ROOT)

TARGET_ROOT = UPLOAD_ROOT / "project"

shutil.copytree(
    SOURCE_ROOT,
    TARGET_ROOT
)

shutil.copytree(
    RESULTS_DIR,
    TARGET_ROOT / "results/training_diagnostics",
    dirs_exist_ok=True
)

assert (
    TARGET_ROOT /
    "results/srgan_quality/quality_summary.json"
).is_file()

assert (
    TARGET_ROOT /
    "results/classifier_comparison/paired_statistics.json"
).is_file()

assert (
    TARGET_ROOT /
    "checkpoints/srgan/groupnorm_v2/resume/ckpt-150.index"
).is_file()

print("\nUploading training history figures...")

try:

    kagglehub.dataset_upload(
        "niklasohlenroth/applied-ai-midterm-srgan-checkpoints",
        str(UPLOAD_ROOT),
        version_notes=(
            "Training history figures and report metadata"
        )
    )

    print("\nTRAINING HISTORY CLOUD BACKUP UPLOADED.")

except Exception as error:

    print("\nCloud upload failed:", repr(error))
    print("Figures remain available locally.")

print("\nSTEP 29 COMPLETE")
